In [13]:
from pathlib import Path 
from uuid import uuid5, NAMESPACE_URL
from dotenv import load_dotenv
load_dotenv() 
import os

from langchain_community.document_loaders import (PyPDFLoader, TextLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openrouter import ChatOpenRouter
from langchain_qdrant import QdrantVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.genai import types
from google import genai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

DATA_DIR = Path("data")
DB_PATH = "qdrant_db"
COLLECTION = "policies"
EMBED_DIM = 1536

# 1. DOC loading + Chunking

In [14]:
documents = []

for data_path in sorted(DATA_DIR.iterdir()):
    if data_path.suffix.lower() == ".pdf":
        loader = PyPDFLoader(str(data_path))
    elif data_path.suffix.lower() == ".txt":
        loader = TextLoader(str(data_path), encoding="utf-8")
    else:
        continue
    
    loaded = loader.load()

    for doc in loaded :
        doc.metadata['source'] = data_path.name
        doc.metadata['document_id'] = data_path.stem
    documents.extend(loaded)
print("LOADED DOCUMENTS : ", len(documents))

LOADED DOCUMENTS :  1


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 250,
    chunk_overlap = 40,
    add_start_index = True
)
chunks  = splitter.split_documents(documents)

for index,chunk in enumerate(chunks):
    chunk.metadata['chunk_index'] = index
    chunk_id = str(uuid5(
        NAMESPACE_URL,
        f"{chunk.metadata['source']}:{chunk.metadata.get('page', 0)}:{chunk.metadata['start_index']}:{chunk.page_content}"
    ))
    
    chunk.id = chunk_id
    chunk.metadata['chunk_id'] = chunk_id

print("Created Chunks : ", len(chunks))

Created Chunks :  4


# 2. Embedding + QDRANT CLIENT

In [18]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document",# Tailors embeddings for document storage
    output_dimensionality=EMBED_DIM
)

actual_dim = len(embeddings.embed_query("dimension check"))
assert actual_dim == EMBED_DIM, (
    f"Expected {EMBED_DIM}, got {actual_dim}"
)

client = QdrantClient(path = DB_PATH)

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size = EMBED_DIM,
        distance=Distance.COSINE
    )
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


True

# LANGCHAIN + QDRANT

In [19]:
vector_store = QdrantVectorStore(
    client = client,
    collection_name=COLLECTION,
    embedding=embeddings
)

vector_store.add_documents(
    documents=chunks,
    ids=[chunk.id for chunk in chunks]
)

print('STORED CHUNKS :', client.count(
    collection_name=COLLECTION,
    exact=True
).count)

client.close()

STORED CHUNKS : 4
